### PyMuPDF for Retrieval in RAG

* To convert PDF to text strings (Markdown)
  

In [16]:
import os
import sys
import pathlib
import gc

import pymupdf4llm
import pymupdf
import fitz

### Path

In [2]:
test_sample_1 = "/home/jerlshin/FieldOfInterest/My_Work/__PROJECTS__/TableExtraction/page_1.pdf"
test_sample_2 = "/home/jerlshin/FieldOfInterest/My_Work/__PROJECTS__/TableExtraction/page_2.pdf"

In [3]:
output = pathlib.Path("../output_images")

for item in output.iterdir():
    if(item.is_file()):
        print(item)

../output_images/page_2.png
../output_images/page_1.png


#### Multimodal LLM Application with PDF data

In [4]:
## To convert the pdf to markdwon

# with chunking
md_text = pymupdf4llm.to_markdown(doc=test_sample_1, page_chunks=True)

# with images
md_text_with_image_extract = pymupdf4llm.to_markdown(doc=test_sample_1, page_chunks=True, write_images=True, image_path="./images")

# embedding images ==> image data in base64 format
bed_images = pymupdf4llm.to_markdown(doc=test_sample_1, page_chunks=True, write_images=True, image_path="./images", embed_images=True)

# extracting words
chunks_words = pymupdf4llm.to_markdown(doc=test_sample_1, page_chunks=True, extract_words=True)

In [5]:
gc.collect()

100

#### Find and Extract Tables from PDFs

In [6]:
# it have list of all the pages
doc = pymupdf.open(test_sample_1)
print(doc.metadata)

# accessing pages
page = doc[0]
print(page)

{'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'pypdf', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None}
page 0 of /home/jerlshin/FieldOfInterest/My_Work/__PROJECTS__/TableExtraction/page_1.pdf


In [7]:

# find tables
find_table = page.find_tables(strategy="text")

# tables
table1 = find_table.tables[0]

table1.extract()

[['', '', '', '', 'Size', 'Recommended', 'Recommended', 'Special'],
 ['', 'IADC', '', 'r', 'ange,', 'WOB, lb/in.', 'rotary speed,', 'features/'],
 ['Bit Name', 'code', '', '', 'in.', 'diameter', 'rpm', 'usage*'],
 ['', '', '', '', '', '', '', ''],
 ['Surface set diamond', 'coring bits', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', ''],
 ['CB17', 'M613', '6-121/4', '', '', '250-3,500', '80-140', 'F'],
 ['', '', '', '', '', '', '', ''],
 ['Medium', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', ''],
 ['Insert roller cone bits', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', ''],
 ['M84', '615', '11 5/8, 17,', '171/2', '', '3,000-7,000', '300-80', 'X'],
 ['M84F',
  '617',
  '43/4, 51/2,',
  '61/4,',
  '61/2, 63/4, 77/8, 83/8,',
  '3,000-7,000',
  '80-40',
  'BX'],
 ['', '', '81/2, 83/4,', '91/2,', '97/8, 105/8, 12, 121/4', '', '', ''],
 ['M84FD',
  '617',
  '83/8, 81/2,',
  '8 3/4,',
  '121/4',
  '3,000-7,000',
  '80-40',
  'BXW'],
 ['MAF', '617', '77/8',

We can use this to identify the rows of the table. 

In [8]:
df = table1.to_pandas()

In [9]:
df

,Col0,Col1,Col2,Col3,4-Size,5-Recommended,6-Recommended,7-Special
0,,IADC,,r,"ange,","WOB, lb/in.","rotary speed,",features/
1,Bit Name,code,,,in.,diameter,rpm,usage*
2,,,,,,,,
3,Surface set diamond,coring bits,,,,,,
4,,,,,,,,
5,CB17,M613,6-121/4,,,"250-3,500",80-140,F
6,,,,,,,,
7,Medium,,,,,,,
8,,,,,,,,
9,Insert roller cone bits,,,,,,,


In [10]:
doc2 = pymupdf.open(test_sample_2)
page2 = doc2[0]

print(page2)

find_table2 = page2.find_tables(strategy="text")
table2 = find_table2.tables[0]
df2 = table2.to_pandas()

df2

page 0 of /home/jerlshin/FieldOfInterest/My_Work/__PROJECTS__/TableExtraction/page_2.pdf


,Col0,Col1,Re,comm,ended,for,these sy,stems,Col8,Col9,Col10,Col11,Pr,od
0,,,Wa,ter-ba,sed Fl,uids,,,,,,,Fu,nc
1,,,,,,,,,,,,,,
2,,,,,,,,,,,,gas,,
3,,,,rsed,,eated,,,Salt,,base,"oam,",,
4,,,,e,d,tr,,ds,d,,c-,", f",,ry
5,,,,-disp,perse,cium-,ymer,Soli,urate,base,theti,mist,mary,onda
6,radename,,Description,Non,Dis,Cal,Pol,Low,Sat,Oil-,Syn,"Air,",Pri,Sec
7,,,,,,,,,,,,,,
8,BARACARB 230,0,Sized calcium carbonate,X,X,X,X,X,X,X,X,,LO,W
9,BARACAT,,Cationic polymer solution,X,,X,X,X,X,,,,SH,FL


#### Text Extraction Techniques

In [11]:
text = page.get_text()

print(text)

Page 10
Surface set diamond coring bits
CB17
M613
6-121/4
250-3,500
80-140
F
Medium
Insert roller cone bits
M84
615
11 5/8, 17, 171/2
3,000-7,000
300-80
X
M84F
617
43/4, 51/2, 61/4, 61/2, 63/4, 77/8, 83/8,
3,000-7,000
80-40
BX
81/2, 83/4, 91/2, 97/8, 105/8, 12, 121/4
M84FD
617
83/8, 81/2, 8 3/4, 121/4
3,000-7,000
80-40
BXW
MAF
617
77/8
3,000-7,000
80-40
AB
MM88
625
121/4
2,500-7,000
150-80
MB
M84CF
627
77/8, 83/4, 11
3,000-7,000
80-40
BY
M84CFD
627
81/2, 83/4
3,000-7,000
14824
BYW
M85F
627
61/2, 63/4, 77/8, 83/4
3,500-7,000
80-40
BX
M86CF
627
77/8
3,000-7,000
80-40
BY
M89T
627
171/2
2,500-7,000
150-80
MB
M89TF
627
57/8, 6, 61/8, 61/4, 61/2, 63/4, 77/8,
3,500-7,000
80-40
BX
M89TFD
627
81/2, 83/4, 91/2
3,500-7,000
80-40
BXW
M89F
637
77/8, 121/4
4,000-7,000
70-40
BY
M89FD
637
97/8
4,000-7,000
70-40
BYW
Size
Recommended
Recommended
Special
IADC
range,
WOB, lb/in.
rotary speed,
features/
Bit Name
code
in.
diameter
rpm
usage*
Security DBS



In [12]:
len(text)

947

Getting blocks

Previously, line breaks are inserted in the output, but no additional information is produced

In [13]:
blocks = page.get_text("blocks", sort=True)

In [14]:
blocks

[(211.19700622558594,
  14.255992889404297,
  274.77490234375,
  23.755992889404297,
  'Security DBS\n',
  7,
  0),
 (469.1999816894531,
  13.55999755859375,
  474.73199462890625,
  34.92084503173828,
  'Page 10\n',
  0,
  0),
 (18.00104522705078,
  33.464229583740234,
  454.5821228027344,
  55.1937370300293,
  'Size\nRecommended\nRecommended\nSpecial\nIADC\nrange,\nWOB, lb/in.\nrotary speed,\nfeatures/\nBit Name\ncode\nin.\ndiameter\nrpm\nusage*\n',
  6,
  0),
 (18.01020050048828,
  63.867000579833984,
  137.56129455566406,
  71.44300079345703,
  'Surface set diamond coring bits\n',
  1,
  0),
 (18.01020050048828,
  76.9530029296875,
  443.01336669921875,
  82.7249984741211,
  'CB17\nM613\n6-121/4\n250-3,500\n80-140\nF\n',
  2,
  0),
 (18.011199951171875,
  92.30191040039062,
  56.20109558105469,
  101.80191040039062,
  'Medium\n',
  3,
  0),
 (18.011199951171875,
  107.66621398925781,
  97.60493469238281,
  115.2422103881836,
  'Insert roller cone bits\n',
  4,
  0),
 (18.01119995117

**We can preprocess it and make a refined table. We can make a stringent pipeline for this.**

* It should recognize all the texts
* Make sure all the texts are recognized with bounding boxes accurately
* Continuous text should be joined with bounding boxes.
* Put that to Dataframe as you process
* Re-define the structure of the table

In [15]:
# Each list element is a block,
# It contains the position, text content, block number and block type

for block in blocks:
    print(block[4])
    print("="*50)

Security DBS

Page 10

Size
Recommended
Recommended
Special
IADC
range,
WOB, lb/in.
rotary speed,
features/
Bit Name
code
in.
diameter
rpm
usage*

Surface set diamond coring bits

CB17
M613
6-121/4
250-3,500
80-140
F

Medium

Insert roller cone bits

M84
615
11 5/8, 17, 171/2
3,000-7,000
300-80
X
M84F
617
43/4, 51/2, 61/4, 61/2, 63/4, 77/8, 83/8,
3,000-7,000
80-40
BX
81/2, 83/4, 91/2, 97/8, 105/8, 12, 121/4
M84FD
617
83/8, 81/2, 8 3/4, 121/4
3,000-7,000
80-40
BXW
MAF
617
77/8
3,000-7,000
80-40
AB
MM88
625
121/4
2,500-7,000
150-80
MB
M84CF
627
77/8, 83/4, 11
3,000-7,000
80-40
BY
M84CFD
627
81/2, 83/4
3,000-7,000
14824
BYW
M85F
627
61/2, 63/4, 77/8, 83/4
3,500-7,000
80-40
BX
M86CF
627
77/8
3,000-7,000
80-40
BY
M89T
627
171/2
2,500-7,000
150-80
MB
M89TF
627
57/8, 6, 61/8, 61/4, 61/2, 63/4, 77/8,
3,500-7,000
80-40
BX
M89TFD
627
81/2, 83/4, 91/2
3,500-7,000
80-40
BXW
M89F
637
77/8, 121/4
4,000-7,000
70-40
BY
M89FD
637
97/8
4,000-7,000
70-40
BYW



In [30]:
for block in blocks:
    print(block)
    print("="*50)

(211.19700622558594, 14.255992889404297, 274.77490234375, 23.755992889404297, 'Security DBS\n', 7, 0)
(469.1999816894531, 13.55999755859375, 474.73199462890625, 34.92084503173828, 'Page 10\n', 0, 0)
(18.00104522705078, 33.464229583740234, 454.5821228027344, 55.1937370300293, 'Size\nRecommended\nRecommended\nSpecial\nIADC\nrange,\nWOB, lb/in.\nrotary speed,\nfeatures/\nBit Name\ncode\nin.\ndiameter\nrpm\nusage*\n', 6, 0)
(18.01020050048828, 63.867000579833984, 137.56129455566406, 71.44300079345703, 'Surface set diamond coring bits\n', 1, 0)
(18.01020050048828, 76.9530029296875, 443.01336669921875, 82.7249984741211, 'CB17\nM613\n6-121/4\n250-3,500\n80-140\nF\n', 2, 0)
(18.011199951171875, 92.30191040039062, 56.20109558105469, 101.80191040039062, 'Medium\n', 3, 0)
(18.011199951171875, 107.66621398925781, 97.60493469238281, 115.2422103881836, 'Insert roller cone bits\n', 4, 0)
(18.011199951171875, 120.75221252441406, 448.6162414550781, 235.7242431640625, 'M84\n615\n11 5/8, 17, 171/2\n3,000

In [31]:
block[:4]  # position of the text

(18.011199951171875, 120.75221252441406, 448.6162414550781, 235.7242431640625)

Getting as dictionary

In [33]:
pgdict = page.get_text("dict", sort=True)

print(pgdict.keys())

dict_keys(['width', 'height', 'blocks'])


In [50]:
for block in pgdict["blocks"]:
    print(block, "\n")
    print("="*50)

{'number': 7, 'type': 0, 'bbox': (211.19700622558594, 14.255992889404297, 274.77490234375, 23.755992889404297), 'lines': [{'spans': [{'size': 10.0, 'flags': 20, 'bidi': 0, 'char_flags': 16, 'font': 'Arial.Bold083.313', 'color': 0, 'alpha': 255, 'ascender': 0.800000011920929, 'descender': -0.20000000298023224, 'text': 'Security DBS', 'origin': (211.19700622558594, 21.595993041992188), 'bbox': (211.19700622558594, 14.255992889404297, 274.77490234375, 23.755992889404297)}], 'wmode': 0, 'dir': (1.0, 0.0), 'bbox': (211.19700622558594, 14.255992889404297, 274.77490234375, 23.755992889404297)}]} 

{'number': 0, 'type': 0, 'bbox': (469.1999816894531, 13.55999755859375, 474.73199462890625, 34.92084503173828), 'lines': [{'spans': [{'size': 6.0, 'flags': 4, 'bidi': 0, 'char_flags': 16, 'font': 'Arial050', 'color': 0, 'alpha': 255, 'ascender': 0.800000011920929, 'descender': -0.20000000298023224, 'text': 'Page 10', 'origin': (470.3999938964844, 13.55999755859375), 'bbox': (469.1999816894531, 13.55

Perform a text extraction on a limited area of the page

In [64]:
page

page 0 of /home/jerlshin/FieldOfInterest/My_Work/__PROJECTS__/TableExtraction/page_1.pdf

In [60]:
cropstart = page.search_for("Security DBS")



In [61]:
for rect in cropstart:
    print(rect)

In [62]:
sf = page.search_for("Security DBS", quads=True)

In [63]:
for rect in sf:
    print(rect)